# Phát hiện vùng biển số bằng xử lý ảnh cổ điển
Notebook hiệu chỉnh bộ phát hiện Sobel/Canny theo kế hoạch XLA. Gaussian, phát hiện biên, phép đóng và lọc hình học chạy bằng OpenCV. Tham số được chọn trên val, không dùng pretrained, checkpoint hay OCR đóng gói.

Dataset hiện có nhãn khung. Đầu ra của notebook là cấu hình phát hiện và số liệu IoU, precision, recall, F1. So khớp ký tự là bước tiếp theo sau khi bổ sung mẫu ký tự tự gán nhãn.

## 1. Thiết lập
Chạy trong môi trường có Jupyter, NumPy và OpenCV. Nếu thiếu thư viện, chạy `%pip install numpy opencv-python-headless` trong một cell riêng. Đặt `DATASET_OVERRIDE` nếu dữ liệu nằm ở vị trí khác. `SMOKE_RUN=True` dùng một mẫu nhỏ để kiểm tra luồng.

In [1]:
import csv
import itertools
import json
import random
from pathlib import Path

import cv2
import numpy as np

DATASET_OVERRIDE = None
SMOKE_RUN = False
SEED = 42
MAX_SIDE = 768
IOU_THRESHOLD = 0.5
OUTPUT_DIR = Path('outputs/detector_smoke' if SMOKE_RUN else 'outputs/detector')

dataset_name = 'License Plate Detection Dataset'
roots = [
    Path(dataset_name),
    Path('../CuoiKy') / dataset_name,
]
kaggle_input = Path('/kaggle/input')
if kaggle_input.is_dir():
    roots.extend(sorted(kaggle_input.rglob(dataset_name)))
if DATASET_OVERRIDE is not None:
    roots = [Path(DATASET_OVERRIDE)]
root = next((p for p in roots if (p / 'images').is_dir() and (p / 'labels').is_dir()), None)
if root is None:
    raise FileNotFoundError('Dat DATASET_OVERRIDE den thu muc co images/ va labels/.')
print(root.resolve())

ModuleNotFoundError: No module named 'cv2'

## 2. Đọc dữ liệu và nhãn
Tọa độ YOLO chuẩn hóa được đổi sang khung `xyxy`. Bỏ các bản trùng pixel đã biết theo bảng EDA và giữ bản val của cặp train/val. Ảnh gần trùng chưa xác minh vẫn là giới hạn của phép đo. Có thể thêm ảnh âm tính đã xác minh vào `data/gate_samples/test_negative/` để đo tỷ lệ khoanh nhầm.

In [ ]:
EXACT_DUPLICATES = {
    'train': {'quandoi64.jpg', 'quandoi92.jpg', 'quandoi99.jpg'},
    'val': set(),
    'test': {'quandoi75.jpg'},
}
IMAGE_SUFFIXES = {'.jpg', '.jpeg', '.png', '.webp', '.bmp'}

def image_paths(directory):
    return sorted(p for p in directory.iterdir() if p.suffix.lower() in IMAGE_SUFFIXES)

def split_paths(split):
    paths = image_paths(root / 'images' / split)
    paths = [p for p in paths if p.name not in EXACT_DUPLICATES[split]]
    if SMOKE_RUN:
        paths = random.Random(SEED).sample(paths, min(18, len(paths)))
    if not paths:
        raise ValueError(f'Tap {split} rong.')
    return paths

def read_gray(path):
    gray = cv2.imdecode(np.fromfile(path, dtype=np.uint8), cv2.IMREAD_GRAYSCALE)
    if gray is None:
        raise ValueError(f'Khong doc duoc: {path}')
    return gray

def ground_truth(path, split, image_shape):
    height, width = image_shape
    relative = path.relative_to(root / 'images' / split).with_suffix('.txt')
    label_path = root / 'labels' / split / relative
    boxes = []
    for row in label_path.read_text(encoding='utf-8-sig').splitlines():
        if not row.strip():
            continue
        label, x, y, w, h = map(float, row.split())
        if label != 0 or w <= 0 or h <= 0:
            raise ValueError(f'Nhan khong hop le: {label_path}')
        box = (
            max(0, (x - w / 2) * width),
            max(0, (y - h / 2) * height),
            min(width, (x + w / 2) * width),
            min(height, (y + h / 2) * height),
        )
        if box[0] >= box[2] or box[1] >= box[3]:
            raise ValueError(f'Khung nam ngoai anh: {label_path}')
        boxes.append(box)
    return boxes

val_paths = split_paths('val')
test_paths = split_paths('test')
print(f'Val: {len(val_paths)} | Test: {len(test_paths)}')

## 3. Bộ phát hiện Sobel và Canny
Lọc Gaussian và trích biên. Phép đóng liên kết các nét gần nhau. Giữ vùng có tỷ lệ, diện tích và độ phủ biên phù hợp, rồi loại các hộp chồng lặp. Các khung trả về theo tọa độ ảnh ban đầu.

In [ ]:
def iou(a, b):
    overlap_w = max(0, min(a[2], b[2]) - max(a[0], b[0]))
    overlap_h = max(0, min(a[3], b[3]) - max(a[1], b[1]))
    intersection = overlap_w * overlap_h
    area_a = (a[2] - a[0]) * (a[3] - a[1])
    area_b = (b[2] - b[0]) * (b[3] - b[1])
    union = area_a + area_b - intersection
    return intersection / union if union else 0.0

def prepare_edges(gray):
    source_height, source_width = gray.shape
    scale = min(1, MAX_SIDE / max(gray.shape))
    if scale < 1:
        size = (max(1, round(source_width * scale)), max(1, round(source_height * scale)))
        gray = cv2.resize(gray, size, interpolation=cv2.INTER_AREA)
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    gradient = cv2.Sobel(blur, cv2.CV_32F, 1, 0, ksize=3)
    sobel = cv2.convertScaleAbs(gradient)
    _, sobel_mask = cv2.threshold(sobel, 0, 255, cv2.THRESH_BINARY | cv2.THRESH_OTSU)
    return {
        'sobel': sobel_mask,
        'canny': cv2.Canny(blur, 50, 150),
    }, (source_width / gray.shape[1], source_height / gray.shape[0])

def candidate_boxes(edges, scale, config):
    height, width = edges.shape
    kernel = cv2.getStructuringElement(cv2.MORPH_RECT, tuple(config['kernel']))
    mask = cv2.morphologyEx(edges, cv2.MORPH_CLOSE, kernel)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    candidates = []
    for contour in contours:
        x, y, w, h = cv2.boundingRect(contour)
        area = w * h
        if w < 14 or h < 8 or not 1.15 <= w / h <= 6.8:
            continue
        if not config['min_area_ratio'] <= area / (width * height) <= 0.35:
            continue
        fill = cv2.contourArea(contour) / area
        edge_density = cv2.countNonZero(edges[y:y+h, x:x+w]) / area
        if fill < 0.25 or edge_density < 0.065:
            continue
        margin = max(1, round(h * 0.08))
        box = (
            max(0, x - margin) * scale[0],
            max(0, y - margin) * scale[1],
            min(width, x + w + margin) * scale[0],
            min(height, y + h + margin) * scale[1],
        )
        candidates.append((box, fill * min(edge_density, 0.35)))
    kept = []
    for box, score in sorted(candidates, key=lambda item: -item[1]):
        if all(iou(box, other) < 0.45 for other, _ in kept):
            kept.append((box, score))
    return kept[:CONFIG_MAX_BOXES]

CONFIG_MAX_BOXES = 5

def detect(gray, config):
    edges, scale = prepare_edges(gray)
    return candidate_boxes(edges[config['method']], scale, config)

## 4. Chọn cấu hình trên val
Ghép mỗi dự đoán với tối đa một nhãn trong cùng ảnh theo IoU ≥ 0,5. Duyệt dự đoán từ điểm cao đến thấp. Dùng F1 micro để chọn một cấu hình cho từng phương pháp và cấu hình cuối cùng.

In [ ]:
def match_counts(predictions, truth):
    available = set(range(len(truth)))
    matched = 0
    for box, _ in predictions:
        if not available:
            break
        index = max(sorted(available), key=lambda i: iou(box, truth[i]))
        if iou(box, truth[index]) >= IOU_THRESHOLD:
            available.remove(index)
            matched += 1
    return np.array([matched, len(predictions) - matched, len(truth) - matched])

def metrics(counts):
    tp, fp, fn = map(int, counts)
    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    f1 = 2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0.0
    return dict(tp=tp, fp=fp, fn=fn, precision=precision, recall=recall, f1=f1)

configs = [
    dict(method=method, kernel=kernel, min_area_ratio=min_area_ratio)
    for method, kernel, min_area_ratio in itertools.product(
        ('sobel', 'canny'), ((9, 3), (15, 5), (21, 7)), (0.0005, 0.002)
    )
]
val_counts = np.zeros((len(configs), 3), dtype=int)
for path in val_paths:
    gray = read_gray(path)
    truth = ground_truth(path, 'val', gray.shape)
    edges, scale = prepare_edges(gray)
    for index, config in enumerate(configs):
        predictions = candidate_boxes(edges[config['method']], scale, config)
        val_counts[index] += match_counts(predictions, truth)

val_rows = [{**config, **metrics(count)} for config, count in zip(configs, val_counts)]
val_rows.sort(key=lambda row: (row['f1'], row['precision'], row['recall']), reverse=True)
best_by_method = {
    method: next(row for row in val_rows if row['method'] == method)
    for method in ('sobel', 'canny')
}
selected = {key: val_rows[0][key] for key in ('method', 'kernel', 'min_area_ratio')}
for row in best_by_method.values():
    print(row)
print('Cau hinh da chon:', selected)

## 5. Chấm test và xuất kết quả
Test chỉ chạy sau khi chốt cấu hình bằng val. So sánh hai nhánh với tham số đã chọn của từng nhánh. Báo riêng tỷ lệ báo nhầm nếu có ảnh âm tính. Khi chưa có ảnh âm tính, kết quả test chỉ đại diện cho các ảnh có biển của dataset.

In [ ]:
test_counts = {method: np.zeros(3, dtype=int) for method in best_by_method}
for path in test_paths:
    gray = read_gray(path)
    truth = ground_truth(path, 'test', gray.shape)
    edges, scale = prepare_edges(gray)
    for method, config in best_by_method.items():
        test_counts[method] += match_counts(candidate_boxes(edges[method], scale, config), truth)

test_rows = [
    dict(method=method, selected_on_val=(method == selected['method']), **metrics(count))
    for method, count in test_counts.items()
]
for row in test_rows:
    print(row)

negative_dir = Path('data/gate_samples/test_negative')
negative_paths = image_paths(negative_dir) if negative_dir.is_dir() else []
negative_report = None
if negative_paths:
    false_images = sum(bool(detect(read_gray(path), selected)) for path in negative_paths)
    negative_report = dict(images=len(negative_paths), false_images=false_images,
                           false_alarm_rate=false_images / len(negative_paths))
    print(negative_report)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
for name, rows in [('validation.csv', val_rows), ('test.csv', test_rows)]:
    with (OUTPUT_DIR / name).open('w', newline='', encoding='utf-8') as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

artifact = {
    'detector': selected,
    'max_side': MAX_SIDE,
    'max_boxes': CONFIG_MAX_BOXES,
    'eval_iou': IOU_THRESHOLD,
    'val_images': len(val_paths),
    'test_images': len(test_paths),
    'negative_test': negative_report,
    'smoke_run': SMOKE_RUN,
}
(OUTPUT_DIR / 'detector_config.json').write_text(
    json.dumps(artifact, indent=2) + '\n', encoding='utf-8'
)
print(OUTPUT_DIR.resolve())

## 6. Kiểm tra một ảnh
Khung xanh là nhãn, khung đỏ là dự đoán. Xem ảnh nhỏ, ảnh nghiêng và ảnh có nhiều biển để xác định ca bỏ sót trước khi triển khai bước cắt biển.

In [ ]:
sample = test_paths[0]
gray = read_gray(sample)
preview = cv2.cvtColor(gray, cv2.COLOR_GRAY2BGR)
truth = ground_truth(sample, 'test', gray.shape)
predictions = detect(gray, selected)
for boxes, color in [(truth, (0, 200, 0)), ([box for box, _ in predictions], (0, 0, 255))]:
    for box in boxes:
        x1, y1, x2, y2 = map(round, box)
        cv2.rectangle(preview, (x1, y1), (x2, y2), color, 2)
preview_path = OUTPUT_DIR / 'preview.png'
success, encoded = cv2.imencode('.png', preview)
if not success:
    raise RuntimeError('Khong tao duoc anh minh hoa.')
encoded.tofile(preview_path)
try:
    from IPython.display import Image, display
    display(Image(filename=str(preview_path), width=640))
except ImportError:
    print(preview_path)